# 00 · Dataset Generation
**CalixAI — Smart Calisthenics Assistant**

Generates a 12,000-row training dataset grounded in published scientific sources:

| Source | Used for |
|--------|----------|
| CDC NHANES 2017–2020 | Height / weight distributions |
| Ainsworth et al., 2011 — *Compendium of Physical Activities* | MET values per exercise |
| NSCA *Essentials of Strength Training* (Haff & Triplett, 2016) | Sets, reps, intensity guidelines |
| IHRSA Global Report 2023 | Goal and fitness-level distributions |
| Karvonen formula | Heart rate estimation |

> Run this notebook **first**, before 01–06.

In [ ]:
import numpy as np
import pandas as pd
import pickle, os
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder

np.random.seed(42)

BASE       = os.path.abspath('..')
MODELS_DIR = os.path.join(BASE, 'models')
RAW_DIR    = os.path.join(BASE, 'data', 'raw')
for d in [MODELS_DIR, RAW_DIR]:
    os.makedirs(d, exist_ok=True)

N = 12000
print(f'Generating {N} rows...')

## Scientific Constants

In [ ]:
EXERCISES      = ['Pull-ups','Push-ups','Dips','Muscle-ups','Plank','Handstand','L-sit','Squats']
FITNESS_LEVELS = ['Beginner','Intermediate','Advanced']
GOALS          = ['Fat Loss','Muscle Gain','Maintenance']
INTENSITIES    = ['Low','Medium','High']
GENDERS        = ['Male','Female']

# Compendium of Physical Activities (Ainsworth et al., 2011)
# DOI: 10.1249/MSS.0b013e31821ece12
EXERCISE_MET = {
    'Pull-ups':8.0, 'Push-ups':8.0, 'Dips':8.0, 'Muscle-ups':10.0,
    'Plank':4.0,    'Handstand':6.0, 'L-sit':5.5, 'Squats':7.5,
}
INTENSITY_FACTOR = {'Low':0.70, 'Medium':1.00, 'High':1.35}
LEVEL_FACTOR     = {'Beginner':0.82, 'Intermediate':1.00, 'Advanced':1.18}

# CDC NHANES 2017-2020 (adults 18-49)
NHANES = {
    'Male':   {'h_mu':175.7,'h_sd':7.2, 'w_mu':89.8,'w_sd':22.0},
    'Female': {'h_mu':162.1,'h_sd':6.8, 'w_mu':77.0,'w_sd':21.5},
}

# IHRSA Global Report 2023 — gym-goer demographics
GOAL_DIST  = [0.38, 0.42, 0.20]   # Fat Loss, Muscle Gain, Maintenance
LEVEL_DIST = [0.45, 0.35, 0.20]   # Beginner, Intermediate, Advanced
GENDER_DIST= [0.55, 0.45]          # Male, Female

# Age distribution weighted by gym-goer demographics
AGE_RANGES  = [(16,20),(20,30),(30,40),(40,50),(50,60)]
AGE_WEIGHTS = [0.10,   0.35,   0.30,   0.20,   0.05]

# NSCA — intensity distribution by fitness level
INTENSITY_DIST = {
    'Beginner':     [0.45, 0.45, 0.10],
    'Intermediate': [0.20, 0.55, 0.25],
    'Advanced':     [0.10, 0.45, 0.45],
}

# NSCA — sets/reps by goal (hypertrophy, fat loss, maintenance ranges)
SETS_REPS = {
    'Fat Loss':    {'sets':(3,5), 'reps':(12,20)},
    'Muscle Gain': {'sets':(3,5), 'reps':(6,12)},
    'Maintenance': {'sets':(2,4), 'reps':(8,15)},
}

# Session duration by goal + level (minutes)
DURATION = {
    ('Fat Loss','Beginner'):(20,40),    ('Fat Loss','Intermediate'):(30,55),    ('Fat Loss','Advanced'):(40,70),
    ('Muscle Gain','Beginner'):(25,45), ('Muscle Gain','Intermediate'):(35,60), ('Muscle Gain','Advanced'):(45,80),
    ('Maintenance','Beginner'):(20,35), ('Maintenance','Intermediate'):(25,45), ('Maintenance','Advanced'):(30,55),
}

# Evidence-based exercise-goal mapping (NSCA + ACE guidelines)
# Weights: primary(50%) / secondary(30%) / tertiary(20%)
EXERCISE_RULES = {
    ('Fat Loss','Beginner'):      [('Push-ups',0.50),('Squats',0.30),('Plank',0.20)],
    ('Fat Loss','Intermediate'):  [('Pull-ups',0.45),('Push-ups',0.30),('Squats',0.25)],
    ('Fat Loss','Advanced'):      [('Pull-ups',0.40),('L-sit',0.35),('Plank',0.25)],
    ('Muscle Gain','Beginner'):   [('Push-ups',0.50),('Dips',0.30),('Squats',0.20)],
    ('Muscle Gain','Intermediate'):[('Pull-ups',0.50),('Dips',0.30),('Squats',0.20)],
    ('Muscle Gain','Advanced'):   [('Muscle-ups',0.50),('Handstand',0.30),('L-sit',0.20)],
    ('Maintenance','Beginner'):   [('Squats',0.50),('Push-ups',0.30),('Plank',0.20)],
    ('Maintenance','Intermediate'):[('Pull-ups',0.45),('Push-ups',0.30),('Squats',0.25)],
    ('Maintenance','Advanced'):   [('Handstand',0.45),('Pull-ups',0.30),('Muscle-ups',0.25)],
}

print('Scientific constants loaded.')

## Generate Dataset

In [ ]:
records = []
for _ in range(N):
    gender  = np.random.choice(GENDERS, p=GENDER_DIST)
    age_idx = np.random.choice(len(AGE_RANGES), p=AGE_WEIGHTS)
    age     = int(np.random.randint(*AGE_RANGES[age_idx]))
    stats   = NHANES[gender]
    height  = round(float(np.clip(np.random.normal(stats['h_mu'], stats['h_sd']), 148, 205)), 1)
    weight  = round(float(np.clip(np.random.normal(stats['w_mu'], stats['w_sd']),  40, 150)), 1)
    bmi     = round(weight / ((height/100)**2), 2)
    goal    = np.random.choice(GOALS,          p=GOAL_DIST)
    level   = np.random.choice(FITNESS_LEVELS, p=LEVEL_DIST)
    intensity = np.random.choice(INTENSITIES,  p=INTENSITY_DIST[level])

    # 96% from weighted rule pool, 4% natural variety
    pool = EXERCISE_RULES[(goal, level)]
    if np.random.random() < 0.96:
        exs, probs = zip(*pool)
        exercise = np.random.choice(exs, p=probs)
    else:
        exercise = np.random.choice(EXERCISES)

    sr       = SETS_REPS[goal]
    sets     = int(np.random.randint(sr['sets'][0], sr['sets'][1]+1))
    reps     = int(np.random.randint(sr['reps'][0], sr['reps'][1]+1))
    duration = int(np.random.randint(*DURATION[(goal, level)]))

    # Karvonen formula — resting HR by fitness level (published norms)
    rhr    = {'Beginner':72, 'Intermediate':65, 'Advanced':58}[level]
    hr_f   = {'Low':0.50, 'Medium':0.65, 'High':0.80}[intensity]
    hr_max = 220 - age
    hr     = int(rhr + (hr_max - rhr) * hr_f)

    # Calorie: MET × weight × time(h) with intensity + level factors
    base  = EXERCISE_MET[exercise] * INTENSITY_FACTOR[intensity] * LEVEL_FACTOR[level] * weight * (duration/60)
    cal   = max(round(base + sets*reps*0.08 + np.random.normal(0, base*0.08), 1), 15.0)

    if goal == 'Fat Loss':
        mp = {'Beginner':'fl_beginner','Intermediate':'fl_intermediate','Advanced':'fl_advanced'}[level]
    elif goal == 'Muscle Gain':
        mp = {'Beginner':'mg_beginner','Intermediate':'mg_intermediate','Advanced':'mg_advanced'}[level]
    else:
        mp = 'mt_standard'

    records.append({
        'age':age, 'gender':gender, 'height_cm':height, 'weight_kg':weight, 'bmi':bmi,
        'fitness_level':level, 'goal':goal, 'exercise':exercise, 'duration_min':duration,
        'intensity':intensity, 'sets':sets, 'reps':reps, 'heart_rate_avg':hr,
        'training_volume':sets*reps, 'effort_score':duration*sets,
        'metabolic_proxy':round(weight/(age+1),3),
        'hr_reserve_ratio':round(hr/hr_max,3),
        'calories_burned':cal, 'meal_plan':mp,
    })

df = pd.DataFrame(records)
df.to_csv(os.path.join(RAW_DIR,'calisthenics_dataset.csv'), index=False)
print(f'Saved: {df.shape[0]} rows x {df.shape[1]} cols')
print(f'Calorie range : {df.calories_burned.min():.0f} – {df.calories_burned.max():.0f} kcal')
print(f'BMI mean      : {df.bmi.mean():.1f} (CDC NHANES avg ~29)')
df.head(3)

## Save Label Encoders

In [ ]:
cat_cols = ['gender','fitness_level','goal','exercise','intensity']
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    le.fit(df[col])
    encoders[col] = le
    print(f'  {col}: {list(le.classes_)}')

with open(os.path.join(MODELS_DIR,'encoders.pkl'),'wb') as f:
    pickle.dump(encoders, f)
print('\nSaved: models/encoders.pkl')

## Validate

In [ ]:
print('=== Dataset Quality ===')
print(f'Missing values : {df.isnull().sum().sum()}')
print(f'BMI            : mean={df.bmi.mean():.1f}  std={df.bmi.std():.1f}')
print(f'Heart rate     : {df.heart_rate_avg.min()}–{df.heart_rate_avg.max()} bpm')
print(f'Calories       : mean={df.calories_burned.mean():.0f}  std={df.calories_burned.std():.0f}')

print('\nExercise assignment (Muscle Gain + Advanced):')
sub = df[(df.goal=='Muscle Gain')&(df.fitness_level=='Advanced')]
print(sub.exercise.value_counts().to_string())

fig, axes = plt.subplots(2, 3, figsize=(15,8))
df['exercise'].value_counts().plot(kind='bar', ax=axes[0][0], color='steelblue', title='Exercise Distribution')
axes[0][0].tick_params(axis='x', rotation=30)
df['goal'].value_counts().plot(kind='bar', ax=axes[0][1], color='salmon', title='Goal Distribution')
df['fitness_level'].value_counts().plot(kind='bar', ax=axes[0][2], color='mediumseagreen', title='Fitness Level')
axes[1][0].hist(df['bmi'], bins=40, color='mediumpurple', edgecolor='white')
axes[1][0].set_title('BMI — CDC NHANES-based')
axes[1][1].hist(df['calories_burned'], bins=40, color='tomato', edgecolor='white')
axes[1][1].set_title('Calories Burned')
axes[1][2].hist(df['age'], bins=20, color='steelblue', edgecolor='white')
axes[1][2].set_title('Age — IHRSA-weighted')
plt.tight_layout()
plt.show()

print('Done. Run notebooks 01 → 06 to train all models.')